# Obtaining the cutouts of the plates

In [1]:
import json
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
import random
import base64
import gzip
import requests
import os

##

In [2]:
index = {}

with open("../Step_2_Integration/observability_bright.json", "rb") as f:
    while True:
        pos = f.tell()
        line = f.readline()

        if not line:
            break

        s = line.strip()

        if s in (b"", b"{", b"}"):
            continue

        # key is before first colon
        key_bytes = s.split(b":", 1)[0]
        key = json.loads(key_bytes.decode())

        index[int(key)] = pos

print("nkeys:", len(index))

def load_observability_key(filename, index, key):
    key = int(key)

    with open(filename, "rb") as f:
        f.seek(index[key])
        line = f.readline().decode().strip()

    if line.endswith(","):
        line = line[:-1]

    rec = json.loads("{" + line + "}")
    data = np.asarray(rec[str(key)])
    return data

nkeys: 51190


In [3]:
mpcnum = 1 #start with Ceres
data = load_observability_key("../Step_2_Integration/observability_bright.json", index, mpcnum)
print(data)

[['243.33128' '-12.933481' '2.2607365' ... '2411433.917859954' 'i00768'
  '7.246961421394394']
 ['230.25215' '-15.6733885' '2.193429' ... '2411565.614578704' 'i01460'
  '7.265207963672594']
 ['339.79263' '-19.649696' '2.4535859' ... '2411898.912928241' 'b06327'
  '7.646977382690467']
 ...
 ['359.98355' '-10.191137' '3.0254352' ... '2447528.49375' 'dnb06498'
  '8.059926445215805']
 ['95.49409' '22.156675' '2.1156383' ... '2447823.74375' 'dnb06687'
  '7.115526937723022']
 ['92.92281' '24.415863' '1.7395307' ... '2447861.7277777777' 'dnb06705'
  '6.666923906544629']]


In [4]:
ra = data[:,0]
dec = data[:,1]
rearth = data[:,2]
rsun = data[:,3]
jd = data[:,4]
plateid = data[:,5]
vmag = data[:,6]

In [7]:
def retrieve_and_save(plateid, ra, dec, solnum=0, path="./"):
    os.makedirs(path, exist_ok=True)

    url = "https://api.starglass.cfa.harvard.edu/public/dasch/dr7/cutout"
    payload = {
        "plate_id": plateid,
        "solution_number": solnum,
        "center_ra_deg": float(ra),
        "center_dec_deg": float(dec),
    }

    r = requests.post(url, json=payload, headers={"Accept": "application/json"})
    r.raise_for_status()

    fits_bytes = gzip.decompress(base64.b64decode(r.json()))

    filename = f"{mpcnum}_{plateid}.fits"
    filepath = os.path.join(path, filename)

    with open(filepath, "wb") as f:
        f.write(fits_bytes)

    return filepath

In [8]:
for i in range(100):
    retrieve_and_save(plateid[i],ra[i],dec[i],path=str(mpcnum))